## **O**ptical **RE**s**ON**ators **PY**thon (**oreonspy**): first basic example

The purpose of this notebook is to provide a first introduction to the **oreonspy** framework and to demonstrate its capabilities in simulating the time-domain dynamics of optical cavities.

This tutorial focuses on the evolution of the intracavity field under time-dependent mirror motion, highlighting key physical effects such as transient behaviour and ring-down.

The user will be guided to get used with the framework through the following steps:

- Cavity definition 
- Simulation initialization
- Step-by-step time evolution.

The tool has some basic methods:
- `cavity()` -> to create the optical cavity to simulate.
- `simulation()` -> to initialize the simulation.
- `sim_step()` -> to propagate one step in the simulation.

### Import dependencies

In [15]:
#Run this cell after uncomment it if you need to install the simulator.
# !pip install oreonspy

In [16]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plt

import oreonspy as op
import oreonspy.utils as ut
op.__version__

'4.4.2'

#### Cavity definition

You can decide the cavity parameters: mirror reflectivities and transmissivities, initial cavity length.

In this case the code is set up to simulate the **Virgo ARM cavity** with initial length of 3000 m

In [17]:
# Set the input electric field, in this case is constant but can be a function of time.
E_in_init = 1. + 0j
lambd = 1064e-9  # m
k = 2.*np.pi / lambd # wave vector

# Definition of the cavity
ARM = op.Cavity(T_A = 0.01377, R_A = 0.986, R_B = 0.99999, cavity_length=3000)
ARM.print_params()


t_a: 0.11734564329364768
r_a: 0.9929753269845127
r_b: 0.9999949999875
cavity_length: 3000


#### Simulation initialization
The following cell is used to initialize the simulation. You can set your **desired sampling frequency** and the method `print_sim_params()` will print the actual sampling frequency which can be an approximation of the desired one.

In [24]:
# Set the calculation frequency of the simulator.
desired_f_calc = 150e3 # 150 kHz

ARM.simulation(lambd=lambd, requested_sampling_frequency=desired_f_calc, initial_input_electric_field=E_in_init)
ARM.print_sim_params()

# Sampling time 
Theta = ARM.sim_params.Theta

wave_number: 5905249.348852994
k2j: (-0-11810498.697705988j)
requested_sampling_frequency: 150000.0
initial_input_electric_field: (1+0j)
sampling_frequency: 149896.229
num_roundtrips: 1
Theta: 6.671281903963041e-06
partial_Theta: False
Theta_fraction: 1.0
num_of_subhist: 3
sampling_frequency_accuracy: 0.9993081933333333


In the next cell we define the critical velocity, the time window and the number of points that we want to simulate. This is done with some methods inside the dependency **utils**.

In [25]:
# Critical velocity of the cavity
v_cr = ut.critical_velocity(cavity=ARM, wavelength=lambd, Finesse = ARM.Finesse)

# Constnat velocity for the linear scan
v = 10.*v_cr

# Generate the time points for the simulation with constant velocity
points, time_window = ut.generate_time_points_for_constant_velocity(velocity=v, 
                                                                    f_calc=ARM.sim_params.sampling_frequency, 
                                                                    number_of_FSR=2)


#### Linear scan above critical velocity


In [26]:
# Define the constant mirror shift step.
shift = v*Theta

# Define the arrays to store the results of the simulation.
E_intra = np.zeros(points, dtype=complex)
E_ref = np.zeros(points, dtype=complex)
error_signal = np.zeros(points) # Pound-Drever-Hall error signal
E_in_init_array = np.full(points, E_in_init, dtype=complex) # Input electric field array

# Define the mirror shift step arrays for the simulation. dz1 is the step for the input mirror, which in this case is stationary, and dz2 is the step for the second mirror.
input_mirror_shift = 0.*shift
output_mirror_shift = np.full(points, shift)

for i in range(points):
    # Step in the simulation and compute the intra cavity field and the reflected field.
    E_intra_value, E_ref_value = ARM.sim_step(output_mirror_displacement=output_mirror_shift[i], input_mirror_displacement=input_mirror_shift, input_electric_field=E_in_init) 
    E_intra[i] = E_intra_value
    E_ref[i] = E_ref_value
    
    # Compute the PDH signal
    error_signal[i] = ut.V_pdh(gamma = 0., E_in = E_in_init, E=E_intra_value)    

fig = ut.plot_cavity_evolution(output_mirror_displacements=output_mirror_shift, 
                               input_electric_field=E_in_init_array, 
                               intracav_electric_field=E_intra, 
                               pdh=error_signal, 
                               reflected_field=E_ref, 
                               file_name="Linear scan of the cavity with constant velocity",
                               save=True)

plt.show()

### A more interesting case

Now we define a smooth velocity inversion using a hyperbolic tangent function, which results in a double crossing resonance with concatenate ringing effect.

It's important to reset the simulation with the method `sim_reset()` if you don't want to call again the `simulation()` method.


In [27]:
# Reset the simulation.
ARM.sim_reset()

# Define the arrays to store the results of the simulation.
E_intra = np.zeros(points, dtype=complex)
E_ref = np.zeros(points, dtype=complex)
error_signal = np.zeros(points) # Pound-Drever-Hall error signal
E_in_init_array = np.full(points, E_in_init, dtype=complex) # Input electric field array

# Definition of the mirror shift with the inversion.
i0 = 12600          # inversion center: the point where the velocity changes sign
w = 300             # width of the transition: the larger, the smoother the velocity inversion

input_mirror_shift = 0.*shift
output_mirror_shift = -shift * np.tanh((np.arange(points) - i0) / w) 

for i in range(points):
    # Step in the simulation and compute the intra cavity field and the reflected field.
    E_intra_value, E_ref_value = ARM.sim_step(output_mirror_displacement=output_mirror_shift[i], input_mirror_displacement=input_mirror_shift, input_electric_field=E_in_init) 
    E_intra[i] = E_intra_value
    E_ref[i] = E_ref_value

    # Compute the PDH signal
    error_signal[i] = ut.V_pdh(gamma = 0., E_in = E_in_init, E=E_intra_value)    

fig = ut.plot_cavity_evolution(output_mirror_displacements=output_mirror_shift, 
                               input_electric_field=E_in_init_array, 
                               intracav_electric_field=E_intra, 
                               pdh=error_signal, 
                               reflected_field=E_ref, 
                               file_name="Oscillating mirror motion with smooth velocity inversion",
                               save=True)

plt.show()